# PATCHR Server on Google Colab

Run the PATCHR server on Colab's free GPU and get a public URL to connect from **PATCHR-Studio**.

Supports **Boltz2** (with inpainting) and **Protenix** models.

**Steps:**
1. Go to **Runtime > Change runtime type > GPU** (T4)
2. Choose a model in cell 4
3. Run all cells below in order
4. Copy the public URL into PATCHR-Studio

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install PATCHR

In [ ]:
!git clone https://github.com/DeepFoldProtein/patchr.git
%cd patchr
!pip install -e . -q

## 3. Install cloudflared

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!cloudflared --version

## 4. Choose model & start server

Pick which model to load at startup:
- **`boltz2`** — Boltz2 with inpainting (default)
- **`protenix`** — Protenix
- **`all`** — Both (uses more VRAM)

In [ ]:
import subprocess
import threading
import re
import time
import requests
import sys

# ── Choose model ──────────────────────────────────────────
MODEL = "boltz2"  # @param ["boltz2", "protenix", "all"]
# ──────────────────────────────────────────────────────────

# Start the PATCHR server via unified CLI
server_process = subprocess.Popen(
    ["patchr", "serve", "--model", MODEL, "--port", "31212", "--device-id", "0"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Stream server logs in background so user can see download/load progress
def stream_server_logs():
    for line in server_process.stdout:
        text = line.decode("utf-8", errors="ignore").rstrip()
        if text:
            print(f"[server] {text}")
log_thread = threading.Thread(target=stream_server_logs, daemon=True)
log_thread.start()

# Wait until server is ready
# Phase 1: Wait for HTTP to respond (server starts immediately, model loads in background)
# Phase 2: Wait for model loading to finish
print(f"Starting PATCHR server (model={MODEL})...")
print("This may take 5-10 minutes on first run (downloading model weights).\n")
server_ready = False
model_ready = False

# Phase 1: Server HTTP ready (should be fast, ~10-30s)
for i in range(60):  # up to 2 minutes
    if server_process.poll() is not None:
        print(f"\nERROR: Server process exited with code {server_process.returncode}")
        print("Check the logs above for details (e.g., out of memory).")
        break
    time.sleep(2)
    try:
        r = requests.get("http://localhost:31212/api/v1/health", timeout=2)
        if r.status_code == 200:
            server_ready = True
            print(f"Server is accepting requests (took ~{(i+1)*2}s)")
            break
    except Exception:
        pass
else:
    print("\nWARNING: Server did not respond within 120s.")

if not server_ready:
    print("Cannot start tunnel — server is not running.")
else:
    # Phase 2: Wait for model to finish loading (can take several minutes on first run)
    print("Waiting for model to finish loading...")
    last_stage = ""
    for i in range(180):  # up to 15 minutes (first download can be slow)
        if server_process.poll() is not None:
            print(f"\nERROR: Server process crashed during model loading.")
            break
        time.sleep(5)
        try:
            r = requests.get("http://localhost:31212/api/v1/health", timeout=3)
            if r.status_code == 200:
                health = r.json()
                if not health.get("model_loading", False):
                    model_ready = True
                    models = health.get("loaded_models", [])
                    elapsed = (i + 1) * 5
                    print(f"\nModel loaded! ({models}) — took ~{elapsed}s")
                    break
                else:
                    stage = health.get("loading_stage", "loading...")
                    elapsed = (i + 1) * 5
                    if stage != last_stage:
                        print(f"  [{elapsed}s] {stage}")
                        last_stage = stage
                    elif i % 12 == 0:  # every 60s repeat current stage
                        print(f"  [{elapsed}s] {stage}")
        except Exception:
            pass
    else:
        print("\nWARNING: Model did not finish loading within 900s.")

    if not model_ready:
        print("Model loading failed or timed out. Check the logs above.")

    # Start cloudflared tunnel (works even while model is loading)
    tunnel_process = subprocess.Popen(
        [
            "cloudflared", "tunnel",
            "--url", "http://localhost:31212",
            "--no-chunked-encoding",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    tunnel_url = None
    def read_output():
        global tunnel_url
        for line in tunnel_process.stderr:
            match = re.search(r"(https://[\w-]+\.trycloudflare\.com)", line.decode("utf-8", errors="ignore"))
            if match:
                tunnel_url = match.group(1)
                break

    t = threading.Thread(target=read_output, daemon=True)
    t.start()
    t.join(timeout=20)

    if tunnel_url:
        # Verify tunnel actually works end-to-end
        tunnel_ok = False
        for attempt in range(3):
            try:
                r = requests.get(f"{tunnel_url}/api/v1/health", timeout=10)
                if r.status_code == 200:
                    tunnel_ok = True
                    break
            except Exception:
                time.sleep(2)

        print("\n" + "=" * 60)
        print(f"  PATCHR Server is running!")
        print(f"  Model: {MODEL}")
        print(f"  Public URL: {tunnel_url}")
        if tunnel_ok:
            print(f"  Status: Verified (health check passed through tunnel)")
        else:
            print(f"  Status: WARNING — tunnel URL obtained but health check failed")
            print(f"          Try the health check cell below, it may need a moment")
        print(f"\n  Copy this URL into PATCHR-Studio")
        print("=" * 60)
    else:
        print("Failed to get tunnel URL. Try re-running this cell.")

## 5. Health check (optional)

Verify the server is reachable through the tunnel.

In [ ]:
import requests

try:
    r = requests.get(f"{tunnel_url}/api/v1/health", timeout=10)
    print("Health check:", r.json())
except Exception as e:
    print(f"Error: {e}")